# Dorado: Two-Stage Reasoning Post-Training

Paper-faithful implementation with configurable experiment profiles.

**Profiles:**
- `smoke` — ~1 min, validates pipeline correctness (SmolLM2-135M)
- `fast` — ~15-30 min, real results on 1.5B (Qwen2.5-Math-1.5B)
- `full` — ~4 hrs, paper-faithful 7B run (Qwen2.5-Math-7B, requires 4-8x L4)

In [ ]:
import os
import getpass
import shutil
import subprocess
from pathlib import Path

# ⚠️ MUST be set BEFORE torch is imported
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Pick a writable runtime cache path (JupyterHub-safe)
repo_candidates = ["/home/jovyan/llmpt", os.getcwd()]
repo_root = next((p for p in repo_candidates if os.path.isdir(p)), os.getcwd())

cache_candidates = [
    os.path.join(repo_root, ".runtime_cache"),
    "/tmp/llmpt_runtime_cache",
]

runtime_cache = None
for cand in cache_candidates:
    try:
        Path(cand).mkdir(parents=True, exist_ok=True)
        test_file = Path(cand) / ".write_test"
        test_file.write_text("ok", encoding="utf-8")
        test_file.unlink(missing_ok=True)
        runtime_cache = cand
        break
    except Exception:
        continue

if runtime_cache is None:
    raise RuntimeError("No writable runtime cache path found.")

os.environ["DORADO_RUNTIME_CACHE"] = runtime_cache
os.environ["HF_HOME"] = f"{runtime_cache}/huggingface"
os.environ["HF_HUB_CACHE"] = f"{runtime_cache}/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = f"{runtime_cache}/huggingface/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{runtime_cache}/huggingface/transformers"
os.environ["PIP_CACHE_DIR"] = f"{runtime_cache}/pip"

# Optional: clear old home cache once to reclaim quota
for stale in ["/home/jovyan/.cache/huggingface", "/home/jovyan/.cache/pip"]:
    if os.path.exists(stale):
        shutil.rmtree(stale, ignore_errors=True)

def pick_free_gpus(min_free_gb: int = 16, max_gpus: int = 8) -> list[str]:
    try:
        cmd = [
            "nvidia-smi",
            "--query-gpu=index,memory.free",
            "--format=csv,noheader,nounits",
        ]
        rows = subprocess.check_output(cmd, text=True).strip().splitlines()
        parsed = []
        for row in rows:
            idx_str, free_mb_str = [x.strip() for x in row.split(",")[:2]]
            parsed.append((idx_str, int(free_mb_str)))

        keep = [x for x in parsed if x[1] >= min_free_gb * 1024]
        if not keep:
            keep = sorted(parsed, key=lambda x: x[1], reverse=True)[:max_gpus]
        else:
            keep = sorted(keep, key=lambda x: x[1], reverse=True)[:max_gpus]

        return [idx for idx, _ in keep]
    except Exception:
        return ["0"]

selected = pick_free_gpus(min_free_gb=16, max_gpus=8)
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(selected)

# Encourage sharding when multiple GPUs are visible
if len(selected) > 1:
    os.environ["DORADO_MAX_MEMORY_PER_GPU"] = "22GiB"

print(f"🧩 CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"📁 Repo root: {repo_root}")
print(f"💾 Runtime cache: {runtime_cache}")

In [ ]:
# 📦 Install / fix dependencies (run once, then restart kernel)
# This ensures torch and transformers are compatible.
%pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -q -U accelerate datasets pyarrow scipy bitsandbytes peft huggingface_hub transformers trl "filelock>=3.12.0"
%pip install -q sympy==1.12 latex2sympy2 word2number regex pebble timeout-decorator
print("✅ Done — restart kernel, then run the cells below.")

In [ ]:
# 🔍 Environment check (run AFTER restarting kernel)
import torch, transformers
print(f"✅ torch {torch.__version__}, transformers {transformers.__version__}")
if torch.cuda.is_available():
    print(f"🎯 GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.device_count()} visible)")
else:
    print("⚠️ No CUDA GPUs available")

In [ ]:
# 🔎 GPU Diagnostic
import subprocess, os

print("="*60)
print("GPU STATUS (nvidia-smi)")
print("="*60)
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import sys, os

# Find the dorado package
print(f"CWD: {os.getcwd()}")
print(f"dorado/ exists here? {os.path.isdir('dorado')}")

candidates = [
    os.path.abspath("."),
    os.path.expanduser("~/llmpt"),
    "/workspaces/llmpt",
]

found = False
for p in candidates:
    if os.path.isdir(os.path.join(p, "dorado")):
        if p not in sys.path:
            sys.path.insert(0, p)
        os.chdir(p)
        print(f"✅ Found dorado at: {p}")
        found = True
        break

if not found:
    print("❌ Could not find dorado/ — add your repo path to 'candidates' above")
    print(f"   sys.path = {sys.path[:5]}")

In [ ]:
# Import the dorado package
%load_ext autoreload
%autoreload 2

from dorado import (
    # config / profiles
    get_profile, PROFILES, build_experiment_grid, make_results_paths,
    # stages (can call individually for debugging)
    run_sft_stage, run_candidate_generation, run_labeling_stage,
    run_dpo_training, run_full_evaluation,
    # orchestrator
    run_single_experiment, run_all_experiments, cleanup_artifacts,
    # utils
    clear_gpu, set_random_seeds, cleanup_storage,
)

print("✅ dorado package loaded")
print(f"Available profiles: {list(PROFILES.keys())}")

## Choose Profile & Configure

| Profile | Time | GPUs | Model | Use Case |
|---------|------|------|-------|----------|
| `smoke` | ~1 min | 1x | SmolLM2-135M | Pipeline validation |
| `fast` | ~15-30 min | 1-2x L4 | Qwen2.5-Math-1.5B | Quick experiments |
| `full` | ~4 hrs | 4-8x L4 | Qwen2.5-Math-7B | Paper-faithful run |

In [ ]:
# ── Choose your profile ────────────────────────────────────────────
PROFILE = "smoke"  # "smoke" | "fast" | "full"

# Optional overrides (uncomment to customize)
# OVERRIDES = {"math_prompt_count": 100, "sft_samples": 500}
OVERRIDES = None

# ── Build experiment grid ──────────────────────────────────────────
EXPERIMENTS = build_experiment_grid(profile=PROFILE, overrides=OVERRIDES)
_, RESULTS_FILE, CHECKPOINT_FILE = make_results_paths()

cfg = EXPERIMENTS[0]
print(f"Profile:      {PROFILE}")
print(f"Base model:   {cfg['base_model']}")
print(f"Finetuning:   {cfg['finetuning_type']}")
print(f"SFT data:     {cfg['sft_dataset_name']} ({cfg['sft_samples']} samples)")
print(f"Math prompts: {cfg['math_prompt_count']}")
print(f"RM strategy:  {cfg['rm_strategy']}")
print(f"DPO β/lr:     {cfg['dpo_beta']}/{cfg['dpo_lr']}")
print(f"Eval engine:  {cfg['eval_engine']}")
print(f"Eval benchmarks: {cfg['eval_benchmarks']}")
print(f"Experiments:  {len(EXPERIMENTS)}")
print(f"Results:      {RESULTS_FILE}")

## Run Experiments

The cell below runs every experiment in the grid, checkpointing after each one.

**Pipeline stages:**
1. **SFT (GC-Boost)** — cold-start on general chat data
2. **Candidate Generation** — self-generated math responses
3. **Preference Labeling** — dual reward (correctness + ArmoRM quality)
4. **DPO Training** — offline optimization on preference pairs
5. **Evaluation** — MATH benchmark with \\boxed{} parsing

**To resume** after a crash/restart: just re-run — completed experiments are skipped via checkpoint.

In [ ]:
results_df = run_all_experiments(EXPERIMENTS, RESULTS_FILE, CHECKPOINT_FILE)

In [ ]:
# Display results
if results_df is not None and not results_df.empty:
    display_cols = [c for c in results_df.columns if 'accuracy' in c or c in ['experiment_id', 'status', 'runtime_minutes', 'improvement_over_base']]
    display(results_df[display_cols])
else:
    print("No results yet.")

In [ ]:
# Cleanup intermediate artifacts (model checkpoints, etc.)
cleanup_artifacts()